# Tahap 4 — Case Solution Reuse


In [78]:
import os
import json
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

import torch
from transformers import AutoTokenizer, AutoModel

import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


In [79]:
BASE_DIR         = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED_FOLDER = os.path.join(BASE_DIR, "data", "processed")
EVAL_FOLDER      = os.path.join(BASE_DIR, "data", "eval")
RESULTS_FOLDER   = os.path.join(BASE_DIR, "data", "results")

os.makedirs(RESULTS_FOLDER, exist_ok=True)

CSV_PATH         = os.path.join(PROCESSED_FOLDER, "cases_clean.csv")
BERT_EMBED_PATH  = os.path.join(PROCESSED_FOLDER, "bert_embeddings.npy")
QUERIES_PATH     = os.path.join(EVAL_FOLDER, "queries.json")
PREDICTIONS_PATH = os.path.join(RESULTS_FOLDER, "predictions.csv")

print("CSV      :", CSV_PATH)
print("Queries  :", QUERIES_PATH)
print("Output   :", PREDICTIONS_PATH)

CSV      : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\cases_clean.csv
Queries  : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\eval\queries.json
Output   : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\results\predictions.csv


Karena notebook ini berdiri sendiri, kita rebuild model TF-IDF dan SVM
dari data yang sudah ada (`cases_clean.csv` dan `bert_embeddings.npy`).

In [80]:
df = pd.read_csv(CSV_PATH, encoding="utf-8")
df["text_full"]       = df["text_full"].fillna("")
df["ringkasan_fakta"] = df["ringkasan_fakta"].fillna("")
df["amar_putusan"]    = df["amar_putusan"].fillna("")

print(f"Jumlah kasus : {len(df)}")
print(f"Kolom        : {list(df.columns)}")

Jumlah kasus : 33
Kolom        : ['case_id', 'nomor_perkara', 'terdakwa', 'pasal_utama', 'pasal_lain', 'tanggal_putusan', 'ringkasan_fakta', 'amar_putusan', 'word_count', 'text_full']


In [81]:
# Buat label
def buat_label(amar: str) -> str:
    amar = str(amar).lower()
    if "menolak" in amar or "tolak" in amar:
        return "tolak"
    elif "mengabulkan" in amar or "kabul" in amar:
        return "kabul"
    else:
        return "lainnya"

df["label"] = df["amar_putusan"].apply(buat_label)
print("Distribusi label:")
print(df["label"].value_counts())

Distribusi label:
label
tolak    29
kabul     4
Name: count, dtype: int64


In [82]:
TEKS_KOLOM = "text_full"
texts      = df[TEKS_KOLOM].tolist()
case_ids   = df["case_id"].tolist()

le = LabelEncoder()
y  = le.fit_transform(df["label"])

X_train_text, X_test_text, y_train, y_test, idx_train, idx_test = train_test_split(
    texts, y, df.index.tolist(),
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train : {len(X_train_text)} | Test : {len(X_test_text)}")

Train : 26 | Test : 7


In [83]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.8, stop_words= None,sublinear_tf=True)
tfidf_vectorizer.fit(X_train_text)
tfidf_matrix_all   = tfidf_vectorizer.transform(texts)
tfidf_matrix_train = tfidf_vectorizer.transform(X_train_text)

svm_model = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm_model.fit(tfidf_matrix_train, y_train)

print("TF-IDF shape:", tfidf_matrix_all.shape)
print("SVM trained  : OK")

TF-IDF shape: (33, 15068)
SVM trained  : OK


In [84]:
from sklearn.preprocessing import normalize
if os.path.exists(BERT_EMBED_PATH):
    bert_embeddings = np.load(BERT_EMBED_PATH)

    bert_embeddings = normalize(bert_embeddings)

    print(f"BERT embeddings dimuat: {bert_embeddings.shape}")

else:
    print("BERT embeddings belum ada, menjalankan encoding ulang...")

    MODEL_NAME = "indobenchmark/indobert-base-p1"

    bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bert_model_obj = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    bert_model_obj.eval()

    def get_bert_embedding(text, max_length=512):
        inputs = bert_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bert_model_obj(**inputs)

        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_embeddings = token_embeddings * attention_mask

        sentence_embedding = (
            masked_embeddings.sum(dim=1)
            / attention_mask.sum(dim=1)
        )

        return sentence_embedding.cpu().numpy().flatten()

    bert_embeddings = np.vstack([
        get_bert_embedding(text)
        for text in texts
    ])

    bert_embeddings = normalize(bert_embeddings)

    np.save(BERT_EMBED_PATH, bert_embeddings)

    print(f"BERT embeddings disimpan: {bert_embeddings.shape}")

BERT embeddings belum ada, menjalankan encoding ulang...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT embeddings disimpan: (33, 768)


In [85]:
case_solutions = {}

for _, row in df.iterrows():
    cid   = row["case_id"]
    label = row["label"]
    amar  = str(row["amar_putusan"]).strip()
    amar_singkat = amar[:300] + "..." if len(amar) > 300 else amar

    case_solutions[cid] = {
        "label":       label,
        "amar_singkat": amar_singkat
    }

print(f"Total solusi tersimpan : {len(case_solutions)}")

sample_key = list(case_solutions.keys())[0]
print(f"\nContoh solusi [{sample_key}]:")
print(f"  Label       : {case_solutions[sample_key]['label']}")
print(f"  Amar singkat: {case_solutions[sample_key]['amar_singkat'][:150]}...")

Total solusi tersimpan : 33

Contoh solusi [case_001]:
  Label       : tolak
  Amar singkat: - menolak permohonan kasasi dari pemohon kasasi/penuntut umum pada kejaksaan negeri kendari tersebut; - memperbaiki putusan pengadilan tinggi sulawesi...


## 3. Fungsi Retrieve 

In [86]:
def retrieve_tfidf(query: str, k: int = 5) -> list:
    query_vec = tfidf_vectorizer.transform([query])
    sims      = cosine_similarity(query_vec, tfidf_matrix_all).flatten()
    top_k_idx = sims.argsort()[::-1][:k]
    return [{"case_id": case_ids[i], "similarity": round(float(sims[i]), 4)} for i in top_k_idx]


def retrieve_svm(query: str, k: int = 5) -> list:
    query_vec     = tfidf_vectorizer.transform([query])
    pred_class    = svm_model.predict(query_vec)[0]
    pred_label    = le.inverse_transform([pred_class])[0]
    same_class_idx = [i for i, c in enumerate(le.transform(df["label"])) if c == pred_class]
    sims           = cosine_similarity(query_vec, tfidf_matrix_all).flatten()
    same_class_sims = sorted([(i, sims[i]) for i in same_class_idx], key=lambda x: x[1], reverse=True)
    top_k_idx = [i for i, _ in same_class_sims[:k]]
    if len(top_k_idx) < k:
        for idx in sims.argsort()[::-1]:
            if idx not in top_k_idx:
                top_k_idx.append(idx)
            if len(top_k_idx) == k:
                break
    return [{"case_id": case_ids[i], "similarity": round(float(sims[i]), 4),
             "pred_label": pred_label} for i in top_k_idx]

def retrieve_bert(query: str, k: int = 5):

    query_emb = get_bert_embedding(query).reshape(1, -1)
    query_emb = normalize(query_emb)

    sims = cosine_similarity(query_emb, bert_embeddings).flatten()

    top_k_idx = sims.argsort()[::-1][:k]

    results = [
        {
            "case_id": case_ids[i],
            "similarity": round(float(sims[i]), 4)
        }
        for i in top_k_idx
    ]

    return results
def bert_get_embedding(text, max_length=512):
    """Wrapper — hanya jalan jika bert_tokenizer & bert_model tersedia."""
    try:
        inputs = bert_tokenizer(text, return_tensors="pt", max_length=max_length,
                                truncation=True, padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = bert_model(**inputs)
        return outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
    except Exception:
        raise NameError("bert_model belum dimuat")


print("Fungsi retrieve siap.")

Fungsi retrieve siap.


## 4. Fungsi Prediksi Solusi



In [87]:
def predict_outcome(
    query: str,
    method: str = "tfidf",
    strategy: str = "majority",
    k: int = 5
) -> dict:
    """
    Prediksi solusi (label amar putusan) untuk kasus baru.

    Parameter:
    ----------
    query    : str  — teks kasus baru
    method   : str  — 'tfidf', 'svm', atau 'bert'
    strategy : str  — 'majority' atau 'weighted'
    k        : int  — jumlah kasus acuan

    Return:
    -------
    dict berisi:
      - predicted_label   : prediksi label amar
      - confidence        : skor kepercayaan (0–1)
      - top_k_cases       : list case_id acuan
      - solution_summary  : ringkasan amar dari kasus paling mirip
    """
    if method == "tfidf":
        top_k = retrieve_tfidf(query, k=k)
    elif method == "svm":
        top_k = retrieve_svm(query, k=k)
    elif method == "bert":
        top_k = retrieve_bert(query, k=k)
    else:
        raise ValueError(f"Method tidak dikenal: {method}")

    top_k_ids = [r["case_id"] for r in top_k]
    
    print("\n==============================")
    print(f"Method : {method}")
    print(f"Strategy : {strategy}")

    for r in top_k:
        cid = r["case_id"]
        print(
            cid,
            round(r["similarity"], 4),
            case_solutions[cid]["label"]
        )
    
    label_scores = {}   
    label_counts = {}  

    for r in top_k:
        cid   = r["case_id"]
        sim   = r["similarity"]
        sol   = case_solutions.get(cid, {})
        label = sol.get("label", "lainnya")

        label_scores[label] = label_scores.get(label, 0.0) + sim
        label_counts[label] = label_counts.get(label, 0)   + 1

    if strategy == "majority":
        predicted_label = max(label_counts, key=label_counts.get)
        total           = sum(label_counts.values())
        confidence      = round(label_counts[predicted_label] / total, 4)

    elif strategy == "weighted":
        predicted_label = max(label_scores, key=label_scores.get)
        total_score     = sum(label_scores.values())
        confidence      = round(label_scores[predicted_label] / total_score, 4) if total_score > 0 else 0.0

    else:
        raise ValueError(f"Strategy tidak dikenal: {strategy}")

    best_case_id      = top_k[0]["case_id"]
    solution_summary  = case_solutions.get(best_case_id, {}).get("amar_singkat", "-")

    return {
        "predicted_label":  predicted_label,
        "confidence":       confidence,
        "top_k_cases":      top_k_ids,
        "solution_summary": solution_summary,
        "label_scores":     label_scores,
        "label_counts":     label_counts
    }

print("Fungsi predict_outcome siap.")

Fungsi predict_outcome siap.


## 5. Demo Manual — 5 Kasus Baru

Jalankan `predict_outcome()` pada 5 query uji dan bandingkan dengan ground truth.

In [88]:
with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    queries = json.load(f)

print(f"Jumlah query uji: {len(queries)}")

Jumlah query uji: 8


In [89]:
print("=" * 70)
print("DEMO MANUAL — TF-IDF + Majority Vote")
print("=" * 70)

for q in queries[:5]:
    result = predict_outcome(q["query_text"], method="tfidf", strategy="majority", k=5)

    print(f"\n[{q['query_id']}]")
    print(f"  Query (50 char) : {q['query_text'][:80]}...")
    print(f"  Prediksi label  : {result['predicted_label']}")
    print(f"  Confidence      : {result['confidence']}")
    print(f"  Top-5 kasus     : {result['top_k_cases']}")
    print(f"  Label counts    : {result['label_counts']}")
    print(f"  Ringkasan solusi: {result['solution_summary'][:120]}...")
    print("-" * 70)

DEMO MANUAL — TF-IDF + Majority Vote

Method : tfidf
Strategy : majority
case_012 0.1177 kabul
case_010 0.0807 tolak
case_031 0.077 tolak
case_008 0.0578 tolak
case_023 0.0516 tolak

[q001]
  Query (50 char) : terdakwa selaku direktur perusahaan dengan sengaja menggelapkan uang kas perusah...
  Prediksi label  : tolak
  Confidence      : 0.8
  Top-5 kasus     : ['case_012', 'case_010', 'case_031', 'case_008', 'case_023']
  Label counts    : {'kabul': 1, 'tolak': 4}
  Ringkasan solusi: − mengabulkan permohonan kasasi dari pemohon kasasi/penuntut umum pada kejaksaan negeri temanggung tersebut; − membatalk...
----------------------------------------------------------------------

Method : tfidf
Strategy : majority
case_012 0.1404 kabul
case_001 0.0748 tolak
case_010 0.0644 tolak
case_004 0.0616 tolak
case_002 0.0585 kabul

[q002]
  Query (50 char) : terdakwa selaku karyawan bagian keuangan terbukti secara sah dan meyakinkan mela...
  Prediksi label  : tolak
  Confidence      : 0.6
  Top-5

## 6. Jalankan Semua Query 

In [90]:
METHOD_LIST   = ["tfidf", "svm", "bert"]
STRATEGY_LIST = ["majority", "weighted"]
K             = 5

all_predictions = []

for q in queries:
    qid   = q["query_id"]
    qtext = q["query_text"]

    for method in METHOD_LIST:
        for strategy in STRATEGY_LIST:
            try:
                result = predict_outcome(qtext, method=method, strategy=strategy, k=K)

                all_predictions.append({
                    "query_id":         qid,
                    "method":           method,
                    "strategy":         strategy,
                    "predicted_label":  result["predicted_label"],
                    "confidence":       result["confidence"],
                    "top_5_case_ids":   str(result["top_k_cases"]),
                    "solution_summary": result["solution_summary"][:200]
                })
            except Exception as e:
                print(f"  [SKIP] {qid} / {method} / {strategy}: {e}")

df_pred = pd.DataFrame(all_predictions)
df_pred.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

print(f"\nTotal prediksi : {len(df_pred)}")
print(f"Disimpan ke    : {PREDICTIONS_PATH}")
df_pred.head(10)


Method : tfidf
Strategy : majority
case_012 0.1177 kabul
case_010 0.0807 tolak
case_031 0.077 tolak
case_008 0.0578 tolak
case_023 0.0516 tolak

Method : tfidf
Strategy : weighted
case_012 0.1177 kabul
case_010 0.0807 tolak
case_031 0.077 tolak
case_008 0.0578 tolak
case_023 0.0516 tolak

Method : svm
Strategy : majority
case_010 0.0807 tolak
case_031 0.077 tolak
case_008 0.0578 tolak
case_023 0.0516 tolak
case_001 0.0496 tolak

Method : svm
Strategy : weighted
case_010 0.0807 tolak
case_031 0.077 tolak
case_008 0.0578 tolak
case_023 0.0516 tolak
case_001 0.0496 tolak

Method : bert
Strategy : majority
case_003 0.4888 tolak
case_031 0.4874 tolak
case_023 0.4757 tolak
case_015 0.4726 tolak
case_014 0.4689 tolak

Method : bert
Strategy : weighted
case_003 0.4888 tolak
case_031 0.4874 tolak
case_023 0.4757 tolak
case_015 0.4726 tolak
case_014 0.4689 tolak

Method : tfidf
Strategy : majority
case_012 0.1404 kabul
case_001 0.0748 tolak
case_010 0.0644 tolak
case_004 0.0616 tolak
case_002 0

,query_id,method,strategy,predicted_label,confidence,top_5_case_ids,solution_summary
0,q001,tfidf,majority,tolak,0.8000,"['case_012', 'case_010', 'case_031', 'case_008...",− mengabulkan permohonan kasasi dari pemohon k...
1,q001,tfidf,weighted,tolak,0.6941,"['case_012', 'case_010', 'case_031', 'case_008...",− mengabulkan permohonan kasasi dari pemohon k...
2,q001,svm,majority,tolak,1.0000,"['case_010', 'case_031', 'case_008', 'case_023...",- menolak permohonan kasasi dari pemohon kasas...
3,q001,svm,weighted,tolak,1.0000,"['case_010', 'case_031', 'case_008', 'case_023...",- menolak permohonan kasasi dari pemohon kasas...
4,q001,bert,majority,tolak,1.0000,"['case_003', 'case_031', 'case_023', 'case_015...",− menolak permohonan kasasi dari pemohon kasas...
5,q001,bert,weighted,tolak,1.0000,"['case_003', 'case_031', 'case_023', 'case_015...",− menolak permohonan kasasi dari pemohon kasas...
6,q002,tfidf,majority,tolak,0.6000,"['case_012', 'case_001', 'case_010', 'case_004...",− mengabulkan permohonan kasasi dari pemohon k...
7,q002,tfidf,weighted,tolak,0.5024,"['case_012', 'case_001', 'case_010', 'case_004...",− mengabulkan permohonan kasasi dari pemohon k...
8,q002,svm,majority,tolak,1.0000,"['case_001', 'case_010', 'case_004', 'case_013...",- menolak permohonan kasasi dari pemohon kasas...
9,q002,svm,weighted,tolak,1.0000,"['case_001', 'case_010', 'case_004', 'case_013...",- menolak permohonan kasasi dari pemohon kasas...


In [91]:
print("=" * 60)
print("RINGKASAN TAHAP 4 — CASE SOLUTION REUSE")
print("=" * 60)
print(f"Total kasus solusi tersimpan : {len(case_solutions)}")
print(f"Total query diproses         : {len(queries)}")
print(f"Metode retrieval             : {METHOD_LIST}")
print(f"Strategi prediksi            : {STRATEGY_LIST}")
print(f"Output predictions.csv       : {PREDICTIONS_PATH}")
print()
print("Distribusi prediksi per metode:")
print(df_pred.groupby(["method", "predicted_label"]).size().unstack(fill_value=0))

RINGKASAN TAHAP 4 — CASE SOLUTION REUSE
Total kasus solusi tersimpan : 33
Total query diproses         : 8
Metode retrieval             : ['tfidf', 'svm', 'bert']
Strategi prediksi            : ['majority', 'weighted']
Output predictions.csv       : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\results\predictions.csv

Distribusi prediksi per metode:
predicted_label  kabul  tolak
method                       
bert                 0     16
svm                  0     16
tfidf                4     12
